# Generate the Old Babylonian letters documentation
This notebook builds the complete **OldBabylonian** example with the public Doc4CF CLI: feature references, indices, structure explorer, corpus diagram, and text/encoding page. Run the cells in order from a Doc4CF checkout. Use a **Python 3.13+ kernel**, Git, and network access for installation/source download.

Source: [Nino-cunei/oldbabylonian](https://github.com/Nino-cunei/oldbabylonian), TF version **1.0.6**, pinned revision `cd8ffe826a598af4715fd724387d9834ec1300d8`. The notebook checks out only that revision in its own workspace. Generated files and caches stay in ignored local directories until the optional publication step.

LLM enrichment is off by default. Set `USE_LLM = True` below to reproduce the enriched build procedure with **openrouter/free**. New responses may differ from the checked-in prose: the free router can choose different models, and generation is not deterministic. DSS and Old Babylonian originally used batched enrichment; this notebook uses the supported one-request-per-feature CLI. All measured pages come from the corpus itself.

## 1. Locate and install Doc4CF
Select the Python 3.13+ kernel before running this cell. The editable install uses the code in this checkout; restart the kernel if your notebook environment asks you to.

In [ ]:
from pathlib import Path
import sys
import subprocess
import shutil
import os
import json

assert sys.version_info >= (3, 13), "Select a Python 3.13+ kernel."
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "doc4cf").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the Doc4CF checkout."
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)
assert shutil.which("git"), "Install Git and restart the kernel."

## 2. Fetch the pinned corpus
Only the base TF version is used, without optional annotation modules. A dedicated clone under `.testdata/example-sources` keeps source data separate from generated HTML. Existing clones are reused only if clean; the revision check prevents silent changes to the source.

In [ ]:
NAME = 'OldBabylonian'
REVISION = 'cd8ffe826a598af4715fd724387d9834ec1300d8'
VERSION = '1.0.6'
SOURCE = ROOT / ".testdata" / "example-sources" / NAME
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/Nino-cunei/oldbabylonian.git", str(SOURCE)], check=True)
assert not subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True).strip(), "Source checkout has local changes."
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", REVISION], check=True)
assert subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip() == REVISION
CORPUS = SOURCE / "tf" / VERSION
assert len(list(CORPUS.glob("*.tf"))) == 67
assert (CORPUS / "otype.tf").is_file() and (CORPUS / "oslots.tf").is_file()

## 3. Choose offline or LLM-enriched output
`USE_LLM = False` builds measured documentation without an API key. For generated descriptions, set it to `True`. The key is read from `OPENROUTER_API_KEY` in the kernel environment, or entered through a hidden prompt; it is never included in the command or notebook output. On Windows, restart Jupyter after setting a persistent user environment variable, or use the prompt.

The full corpus has 67 feature files, so an uncached enriched run makes 67 requests. Each request is independent; there is no shared conversation. The cache reuses successful requests. Free service quotas and availability can interrupt a run: rerun the generation cell after the limit resets. No paid fallback is configured. Review the generated prose before publication.

`--model-attribution feature` credits the responding model on each feature. Change it to `overview` or `none` to change the display without changing the cached prose.

This example has no additional editorial context. To add your own, set `CONTEXT` to a UTF-8 text file; it will be sent with each feature profile.

In [ ]:
USE_LLM = False
CONTEXT = None
OUTPUT = ROOT / ".testdata" / "example-builds" / NAME / ("enriched" if USE_LLM else "offline")
CACHE = ROOT / ".doc4cf-cache" / 'oldbabylonian'
command = [sys.executable, "-m", "doc4cf", str(CORPUS), str(OUTPUT),
           "--project-home", "../../index.html"]
if USE_LLM:
    from getpass import getpass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    assert os.environ.get("OPENROUTER_API_KEY"), "An API key is required for enrichment."
    command += ["--provider", "openrouter", "--model", "openrouter/free",
                "--model-attribution", "feature", "--cache", str(CACHE)]
    if CONTEXT is not None:
        assert CONTEXT.is_file()
        command += ["--dataset-context", str(CONTEXT)]

## 4. Generate all documentation
The subprocess uses the same Python as the notebook kernel. An unsuccessful run raises an error: do not publish its partial output. Rerunning this cell reuses successful cached descriptions. The footer path is intended for the final `examples/<corpus>/` location; it will resolve after publication.

In [ ]:
subprocess.run(command, cwd=ROOT, check=True)
# Preserve the upstream license alongside the generated documentation.
shutil.copyfile(SOURCE / 'LICENSE', OUTPUT / 'DATA-LICENSE.txt')
print("Generated:", OUTPUT / "index.html")

## 5. Check completeness and inspect the pages
Every TF filename must have its own page under `features/`. The JSON must contain its corresponding feature node. Open the overview, structure and diagram pages in a browser, then sample feature descriptions for accuracy. Counts and links can be reproduced; newly generated language should always be reviewed.

In [ ]:
features = {p.stem for p in CORPUS.glob("*.tf")}
assert {p.stem for p in (OUTPUT / "features").glob("*.html")} == features
graph = json.loads((OUTPUT / "knowledge_graph.json").read_text(encoding="utf-8"))
assert {"feature::" + f for f in features} <= graph["nodes"].keys()
for page in ("index.html", "features.html", "structure.html", "diagram.html", "text.html"):
    assert (OUTPUT / page).is_file(), page
for name in features:
    html = (OUTPUT / "features" / (name + ".html")).read_text(encoding="utf-8")
    assert ("Generated description" in html) == USE_LLM
print(f"Verified {len(features)} feature pages and five overview/browser pages.")
print((OUTPUT / "index.html").as_uri())

## 6. Optional: replace the checked-in example
Set `PUBLISH = True` only after reviewing the output. This copies the complete build into `examples/<corpus>/`, replacing that example's existing pages (an offline build removes its LLM prose). It makes no Git commit or push. The project homepage already links to that location, and the Pages workflow publishes the checked-in files without calling any LLM.

Inspect `git diff` before committing. Clear notebook outputs before saving/sharing; keep credentials and the local response cache out of Git.

In [ ]:
PUBLISH = False
if PUBLISH:
    destination = ROOT / "examples" / NAME
    shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
    print("Updated:", destination)
else:
    print("Review build retained locally; checked-in example unchanged.")